# MRC → QC figures → point cloud (`.npy`) → Chroma shape sampling

This notebook mirrors the logic in `run_mrc_chroma_shape.py`:

1. Load `configs/mrc_chroma_shape.yaml` (same parameters as the CLI).
2. Binarize the map, optionally adaptive binning, build XYZ points.
3. Write orthogonal QC PNGs and `module1_2_15A_final_density.mrc`.
4. Save `module1_2_15A_points.npy`, then reload it (to show the offline workflow).
5. Run `ShapeConditioner` + `Chroma.sample` and export PDB/CIF.

**Requirements:** Run Jupyter with the **repository root** as the working directory so imports and `projects/...` paths resolve. The first **code** cell sets `KMP_DUPLICATE_LIB_OK` / `OMP_NUM_THREADS` in `os.environ` *before* importing `run_mrc_chroma_shape` (which loads Chroma). Use **Run All** starting from that code cell.

### Setup: config + imports

The **next code cell** sets OpenMP-related env vars, then loads `configs/mrc_chroma_shape.yaml` and helpers from `run_mrc_chroma_shape.py`.

In [ ]:
import os
import sys
import json
import warnings
from pathlib import Path
from typing import Any, Dict, Optional

# Before importing run_mrc_chroma_shape (loads Chroma / torch).
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")

import numpy as np
import yaml

import mrc_pointcloud as mpc
import qc_orthogonal_plots as qc_plots

REPO_ROOT = Path.cwd().resolve()
_CFG = REPO_ROOT / "configs" / "mrc_chroma_shape.yaml"
if not _CFG.is_file():
    raise FileNotFoundError(
        f"Expected config at {_CFG}. Set the notebook working directory to the repository root."
    )
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from run_mrc_chroma_shape import (
    ADAPTIVE_MASK_FOREGROUND_THRESHOLD,
    CHROMA_SAMPLE_DEFAULTS,
    SHAPE_CONDITIONER_DEFAULTS,
    _build_chroma_sample_kwargs,
    _build_shape_conditioner_kwargs,
    _merge_dict_defaults,
    _save_sampled_proteins,
)

with open(_CFG, "r", encoding="utf-8") as f:
    cfg: Dict[str, Any] = yaml.safe_load(f)

input_mrc = Path(str(cfg["input_mrc"]))
if not input_mrc.is_absolute():
    input_mrc = (REPO_ROOT / input_mrc).resolve()
output_dir = Path(str(cfg["output_dir"]))
if not output_dir.is_absolute():
    output_dir = (REPO_ROOT / output_dir).resolve()
output_dir.mkdir(parents=True, exist_ok=True)

threshold_raw = cfg.get("threshold")
threshold_quantile = float(cfg.get("threshold_quantile", 0.995))
use_adaptive_bin = bool(cfg.get("use_adaptive_bin", True))
max_voxels = int(cfg.get("max_voxels", 2000))
max_bin_rounds = int(cfg.get("max_bin_rounds", 32))
adaptive_pooling = str(cfg.get("adaptive_pooling", "binary_max")).strip().lower()
if adaptive_pooling not in ("binary_max", "mean_fft"):
    raise ValueError('adaptive_pooling must be "binary_max" or "mean_fft".')
subsample_seed = cfg.get("adaptive_subsample_seed", cfg.get("seed", 0))
seed = int(cfg.get("seed", 0))
max_points_conditioner = cfg.get("max_points_conditioner")
conditioner_seed = cfg.get("conditioner_subsample_seed", seed)

shape_conditioner_cfg = _merge_dict_defaults(
    SHAPE_CONDITIONER_DEFAULTS,
    cfg.get("shape_conditioner") if isinstance(cfg.get("shape_conditioner"), dict) else None,
)
chroma_sample_cfg = _merge_dict_defaults(
    CHROMA_SAMPLE_DEFAULTS,
    cfg.get("chroma_sample") if isinstance(cfg.get("chroma_sample"), dict) else None,
)

print("REPO_ROOT:", REPO_ROOT)
print("input_mrc:", input_mrc)
print("output_dir:", output_dir)

## 1) MRC → binary mask → point cloud

Same thresholding and adaptive cap as the shell script.

In [ ]:
data, voxel_size, origin, header = mpc.read_mrc(str(input_mrc))
stats = mpc.inspect_mrc(data, quantiles=(0.5, 0.9, 0.95, 0.99, threshold_quantile))

if threshold_raw is None:
    if use_adaptive_bin:
        warnings.warn(
            "threshold is null: using threshold_quantile for a scalar threshold.",
            stacklevel=1,
        )
    threshold = float(np.quantile(data.astype(np.float64), threshold_quantile))
else:
    threshold = float(threshold_raw)

mask_data = (data >= threshold).astype(np.float32)

adaptive_meta: Optional[Dict[str, Any]] = None
if use_adaptive_bin:
    points, adaptive_meta, final_density = mpc.mrc_to_point_cloud_adaptive_cap(
        data=mask_data,
        voxel_size=voxel_size,
        origin=origin,
        threshold=ADAPTIVE_MASK_FOREGROUND_THRESHOLD,
        max_voxels=max_voxels,
        max_rounds=max_bin_rounds,
        subsample_seed=None if subsample_seed is None else int(subsample_seed),
        adaptive_pooling=adaptive_pooling,
    )
    final_voxel_size = tuple(float(v) for v in adaptive_meta["final_voxel_size_xyz"])
else:
    points = mpc.mrc_to_point_cloud(
        data=mask_data,
        voxel_size=voxel_size,
        origin=origin,
        threshold=ADAPTIVE_MASK_FOREGROUND_THRESHOLD,
        max_points=None,
        random_seed=None,
    )
    final_density = mask_data
    final_voxel_size = (float(voxel_size[0]), float(voxel_size[1]), float(voxel_size[2]))

print("points shape (N, 3):", points.shape, "dtype:", points.dtype)
print("stats keys:", list(stats.keys())[:5], "...")

## 2) QC orthogonal figures + final density MRC

Output filenames match `run_mrc_chroma_shape.py` under `output_dir`.

In [ ]:
qc_input_density_png = output_dir / "module1_2_15A_input_density_orthogonal.png"
qc_input_density_thr0_png = (
    output_dir / "module1_2_15A_input_density_ge_threshold_else_zero_orthogonal.png"
)
qc_input_mask_png = output_dir / "module1_2_15A_input_mask_orthogonal.png"
final_density_mrc = output_dir / "module1_2_15A_final_density.mrc"
qc_density_png = output_dir / "module1_2_15A_final_density_orthogonal.png"
qc_points_png = output_dir / "module1_2_15A_points_orthogonal.png"
qc_overlay_final_png = output_dir / "module1_2_15A_overlay_points_on_final.png"
qc_overlay_input_mask_png = output_dir / "module1_2_15A_overlay_points_on_input_mask.png"

qc_plots.plot_input_density_orthogonal(
    data, voxel_size=voxel_size, origin=origin, out_path=qc_input_density_png
)
qc_plots.plot_input_density_hard_threshold_orthogonal(
    data, threshold=threshold, voxel_size=voxel_size, origin=origin, out_path=qc_input_density_thr0_png
)
qc_plots.plot_input_mask_orthogonal(
    mask_data, voxel_size=voxel_size, origin=origin, out_path=qc_input_mask_png
)

mpc.write_mrc(
    str(final_density_mrc),
    final_density,
    voxel_size=final_voxel_size,
    origin=origin,
    overwrite=True,
)
qc_plots.plot_final_density_orthogonal(
    final_density, voxel_size=final_voxel_size, origin=origin, out_path=qc_density_png
)
qc_plots.plot_points_orthogonal_slabs(
    points,
    grid_shape_zyx=final_density.shape,
    voxel_size=final_voxel_size,
    origin=origin,
    out_path=qc_points_png,
)
qc_plots.plot_overlay_points_on_final_volume(
    final_density,
    voxel_size=final_voxel_size,
    origin=origin,
    points=points,
    out_path=qc_overlay_final_png,
)
qc_plots.plot_overlay_points_on_input_mask(
    mask_data,
    voxel_size_input=voxel_size,
    origin=origin,
    points=points,
    out_path=qc_overlay_input_mask_png,
    voxel_size_final_for_slab=final_voxel_size,
)

qc_paths = [
    qc_input_density_png,
    qc_input_density_thr0_png,
    qc_input_mask_png,
    qc_density_png,
    qc_points_png,
    qc_overlay_final_png,
    qc_overlay_input_mask_png,
]
for p in qc_paths:
    print(p.name, "->", p.resolve())

In [ ]:
from IPython.display import Image, display

# Inline preview of a subset of figures (requires PNG files on disk).
for p in [
    qc_input_density_png,
    qc_input_mask_png,
    qc_points_png,
    qc_overlay_final_png,
]:
    display(Image(filename=str(p)))

## 3) Save point cloud `.npy` (and optional metrics JSON)

In [ ]:
points_npy = output_dir / "module1_2_15A_points.npy"
np.save(points_npy, points)

metrics = {
    "input_mrc": str(input_mrc),
    "shape_zyx": list(data.shape),
    "voxel_size_xyz": list(voxel_size),
    "origin_xyz": list(origin),
    "threshold": threshold,
    "selected_points": int(points.shape[0]),
    "stats": stats,
    "adaptive_meta": adaptive_meta,
}
with (output_dir / "module1_2_15A_pointcloud_metrics.json").open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, sort_keys=True, default=str)
    f.write("\n")

print("Saved:", points_npy.resolve())

## 4) Load `.npy` (offline / next session)

Use this path in a fresh kernel after step 3, or to verify reload.

In [ ]:
points_loaded = np.load(points_npy)
assert points_loaded.shape[1] == 3
points_for_chroma = points_loaded

# Optional cap (same semantics as YAML `max_points_conditioner`).
if max_points_conditioner is not None:
    cap = int(max_points_conditioner)
    if cap <= 0:
        raise ValueError("max_points_conditioner must be positive when set.")
    if points_for_chroma.shape[0] > cap:
        rng = np.random.default_rng(seed=int(conditioner_seed))
        idx = rng.choice(points_for_chroma.shape[0], size=cap, replace=False)
        points_for_chroma = points_for_chroma[idx]

print("Loaded array:", points_loaded.shape, "| passed to conditioner:", points_for_chroma.shape)

## 5) Chroma: `ShapeConditioner` + `sample`

Uses merged `shape_conditioner` / `chroma_sample` from the same YAML as the CLI. **This step downloads/loads Chroma weights and is slow.**

In [ ]:
import torch
from chroma import Chroma, api, conditioners

api_key = cfg.get("api_key")
if api_key:
    api.register_key(str(api_key))

device = "cuda" if torch.cuda.is_available() else "cpu"
chroma = Chroma()

sc_kw = _build_shape_conditioner_kwargs(shape_conditioner_cfg)
conditioner = conditioners.ShapeConditioner(
    points_for_chroma,
    chroma.backbone_network.noise_schedule,
    **sc_kw,
).to(device)

torch.manual_seed(seed)
sample_kw = _build_chroma_sample_kwargs(chroma_sample_cfg, conditioner, device)
sample_out = chroma.sample(**sample_kw)
if sample_kw.get("full_output"):
    shaped_protein, _full_out_dict = sample_out
else:
    shaped_protein = sample_out

out_pdb = output_dir / "module1_2_15A_shape_design.ipynb_example.pdb"
out_cif = output_dir / "module1_2_15A_shape_design.ipynb_example.cif"
saved = _save_sampled_proteins(shaped_protein, out_pdb, out_cif)
print("device:", device)
print(saved)